# 9.10 Ensemble Optimisation

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.10_ensemble-optimisation.ipynb)

Companion code for Section 9.10 of the book. Run the cells in order from top to bottom.

**Example 9.10.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    BaggingClassifier, AdaBoostClassifier, ExtraTreesClassifier,
    StackingClassifier, VotingClassifier
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import time
import warnings; warnings.filterwarnings('ignore')

datasets = {
    'Breast Cancer': load_breast_cancer(),
    'Wine':          load_wine(),
    'Digits':        load_digits(),
}

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Extra Trees':         ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=100, eval_metric='logloss',
                                              use_label_encoder=False, random_state=42, verbosity=0),
    'LightGBM':            lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost':            CatBoostClassifier(iterations=100, random_seed=42, verbose=0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('=== Ensemble Algorithm Benchmark ===')
print(f'{'Algorithm':<22} {'Breast Cancer':>15} {'Wine':>8} {'Digits':>8} {'Avg':>8} {'Time(s)':>8}')
print('='*72)

for name, model in models.items():
    row = {}
    t0  = time.time()
    for ds_name, dataset in datasets.items():
        X_sc = StandardScaler().fit_transform(dataset.data)
        y    = dataset.target
        scores = cross_val_score(model, X_sc, y, cv=cv, scoring='accuracy', n_jobs=-1)
        row[ds_name] = scores.mean()
    elapsed = time.time() - t0
    row['avg'] = np.mean(list(row.values()))
    row['time']= elapsed
    results[name] = row
    print(f'{name:<22} {row["Breast Cancer"]:>15.4f} {row["Wine"]:>8.4f} {row["Digits"]:>8.4f} {row["avg"]:>8.4f}
{row["time"]:>8.2f}')

# ── Ensemble diversity analysis ───────────────────────────────────
print('\n=== Ensemble Diversity (Disagreement between models) ===')
bc   = load_breast_cancer()
X_bc = StandardScaler().fit_transform(bc.data)
y_bc = bc.target

# Get predictions from all models
all_preds = {}
for name, model in list(models.items())[:6]:  # First 6 for clarity
    from sklearn.base import clone
    m = clone(model)
    m.fit(X_bc[:400], y_bc[:400])
    all_preds[name] = m.predict(X_bc[400:])

# Disagreement matrix
names_list = list(all_preds.keys())
n_models   = len(names_list)
disagree   = np.zeros((n_models, n_models))
for i, n1 in enumerate(names_list):
    for j, n2 in enumerate(names_list):
        disagree[i,j] = np.mean(all_preds[n1] != all_preds[n2])

disagree_df = pd.DataFrame(disagree, index=names_list, columns=names_list)
print('\nDisagreement rates (higher = more diverse = better ensemble):')
print(disagree_df.round(4))

# ── Best ensemble practices ───────────────────────────────────────
print('\n=== Ensemble Best Practices ===')
practices = [
    '1. Diversity first: combine algorithms from different families (tree + linear + instance-based)',
    '2. Use soft voting over hard voting when models output probabilities',
    '3. For stacking: use out-of-fold predictions to avoid leakage',
    '4. Prune: remove models with disagreement < 0.05 (too similar = no benefit)',
    '5. Simple blends often beat complex meta-learners - try weighted average first',
    '6. XGBoost + LightGBM + CatBoost + RandomForest = strong baseline ensemble',
    '7. Always validate ensemble on held-out test set, not just CV',
    '8. More base models helps up to ~5-10; diminishing returns after that',
]
for practice in practices:
    print(f'  {practice}')

# ── Final comparison visualisation ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Ensemble Algorithm Comparison', fontsize=14, fontweight='bold')

ax = axes[0]
res_df = pd.DataFrame(results).T
res_df[['Breast Cancer','Wine','Digits']].plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_ylabel('5-Fold CV Accuracy'); ax.set_title('Algorithm Accuracy Across 3 Datasets')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0.85, 1.01)

ax = axes[1]
import seaborn as sns
sns.heatmap(disagree_df, ax=ax, annot=True, fmt='.3f', cmap='YlOrRd',
           linewidths=0.5, vmin=0, vmax=0.3)
ax.set_title('Model Disagreement Matrix\n(higher = more diverse)')

plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
```